# Credit Scoring Model

This notebook trains the model used by the Streamlit app. It uses the available `data/train.csv` file, prepares the same input fields accepted by the app, evaluates a linear SVC classifier, and saves the trained pipeline as `best_model.pkl`.


In [ ]:
import pickle

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC


## Load Data


In [ ]:
df = pd.read_csv("data/train.csv")
df.head()


In [ ]:
df.info()


## Feature Engineering


In [ ]:
df["TotalIncome"] = df["ApplicantIncome"] + df["CoapplicantIncome"]
df["Loan_Status"] = df["Loan_Status"].map({"Y": 1, "N": 0})

features = ["Gender", "Married", "Education", "LoanAmount", "TotalIncome"]
target = "Loan_Status"

for column in ["Gender", "Married", "Education"]:
    df[column] = df[column].fillna(df[column].mode()[0])

df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())

X = df[features]
y = df[target]

X.head()


## Train Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape


## Preprocessing And Model Pipeline


In [ ]:
numeric_features = ["LoanAmount", "TotalIncome"]
categorical_features = ["Gender", "Married", "Education"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", SVC(C=0.1, kernel="linear", probability=True)),
    ]
)

model


## Train And Evaluate


In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Rejected", "Approved"], zero_division=0))


## Prediction Example


In [ ]:
sample = pd.DataFrame(
    {
        "Gender": ["Male"],
        "Married": ["Yes"],
        "Education": ["Graduate"],
        "LoanAmount": [160],
        "TotalIncome": [6000],
    }
)

prediction = model.predict(sample)[0]
probability = model.predict_proba(sample)[0]

print("Prediction:", "Approved" if prediction == 1 else "Rejected")
print(f"Approval Probability: {probability[1] * 100:.2f}%")
print(f"Rejection Probability: {probability[0] * 100:.2f}%")


## Save Model


In [ ]:
with open("best_model.pkl", "wb") as file:
    pickle.dump(model, file)

print("Saved model to best_model.pkl")
